# Question 7
Locating the individual named "Hosseini Mahdavi" and extracting the related residence information.

In [ ]:
# =========================================
# Question 7
# =========================================

# خواندن دیتاست

import pandas as pd

bnd = pd.read_csv(
    'Bank NIT DB.csv',
    encoding='utf-8-sig'
)

# نرمال‌سازی ستون نام و نام خانوادگی
# برای جلوگیری از خطاهای ناشی از تفاوت یونیکد فارسی و عربی
bnd['FULL_NAME'] = (
    bnd['FULL_NAME']
        
    .str.replace('ي', 'ی')
    
    .str.replace('ك', 'ک')
    
    .str.replace(r'\s+', ' ', regex=True)
    
    # حذف فاصله ابتدا و انتهای متن
    .str.strip()
)

target_name = 'حسینی مهدوی'

# ساخت فیلتر برای پیدا کردن فرد
# regex=False باعث می‌شود متن دقیق جستجو شود
mask = bnd['FULL_NAME'].str.contains(
    target_name,
    regex=False,
    na=False
)

# استخراج اطلاعات فرد پیدا شده
result = bnd.loc[
    mask,
    ['FULL_NAME', 'CITY_NAME', 'PROVINCE_NAME']
]
result


# Question 8
Counting the total number of customers whose first name is exactly "Vania".

In [ ]:
# =========================================
# Question 8
# =========================================

import pandas as pd

bnd = pd.read_csv(
    'Bank NIT DB.csv',
    encoding='utf-8-sig'
)

# نرمال‌سازی نام‌ها
# برای جلوگیری از مشکلات مربوط به کاراکترهای فارسی و عربی
bnd['FULL_NAME'] = (
    bnd['FULL_NAME']
    
    .str.replace('ي', 'ی')
    
    .str.replace('ك', 'ک')
    
    .str.replace(r'\s+', ' ', regex=True)
    
    # حذف فاصله ابتدا و انتهای متن
    .str.strip()
)

# جدا کردن بخش‌های مختلف نام
# مثال:
# ['محمد', 'رضایی']
name_parts = bnd['FULL_NAME'].str.split()

# استخراج نام کوچک افراد
# اگر نام با «سیده» شروع شد،
# کلمه دوم به عنوان نام اصلی در نظر گرفته می‌شه
first_name = name_parts.apply(
    lambda x:
    
    # اگر نام شامل «سیده» باشد
    x[1] if len(x) > 1 and x[0] == 'سیده'
    
    # در غیر این صورت اولین کلمه نام کوچک است
    else x[0] if len(x) > 0
    
    # اگر مقدار خالی باشد
    else ''
)

# شمارش تعداد افرادی که نام کوچک آن‌ها «وانیا» است
count_vania = (first_name == 'وانیا').sum()
count_vania

# Question 9
Comparing customers born between 1290–1299 and 1390–1399 and visualizing the population difference using a comparative chart.

In [ ]:
# =========================================
# Question 9
# =========================================

import pandas as pd
import matplotlib.pyplot as plt

bnd = pd.read_csv(
    'Bank NIT DB.csv',
    encoding='utf-8-sig'
)

# تبدیل تاریخ تولد به رشته
# برای جلوگیری از خطاهای پردازشی
bnd['BIRTH_DATE'] = (
    bnd['BIRTH_DATE']
    .astype(str)
    .str.strip()
)

# استخراج سال تولد
bnd['BIRTH_YEAR'] = (
    bnd['BIRTH_DATE']
    .str[:4]
)

# تبدیل سال تولد به عدد
# خطاهای احتمالی به NaN تبدیل می‌شوند
bnd['BIRTH_YEAR'] = pd.to_numeric(
    bnd['BIRTH_YEAR'],
    errors='coerce'
)

# پیدا کردن متولدین 1290 تا 1299
group_1290 = bnd[
    (bnd['BIRTH_YEAR'] >= 1290) &
    (bnd['BIRTH_YEAR'] <= 1299)
]

# پیدا کردن متولدین 1390 تا 1399
group_1390 = bnd[
    (bnd['BIRTH_YEAR'] >= 1390) &
    (bnd['BIRTH_YEAR'] <= 1399)
]

# شمارش تعداد افراد هر گروه
count_1290 = len(group_1290)
count_1390 = len(group_1390)

# رسم نمودار مقایسه‌ای
plt.figure(figsize=(8, 5))

plt.bar(
    ['1290-1299', '1390-1399'],
    [count_1290, count_1390]
)

# عنوان نمودار
plt.title('Comparison of Birth Year Groups')

# عنوان محور افقی
plt.xlabel('Birth Year Range')

# عنوان محور عمودی
plt.ylabel('Population Count')

# نمایش نمودار
plt.show()

# Question 10
Finding customers whose birth date exactly matches the user's birth date (birthday twin search).

In [ ]:
# =========================================
# Question 10
# =========================================

import pandas as pd

# فقط ستون‌های مورد نیاز خوانده می‌شوند
# برای کاهش مصرف حافظه
columns_to_load = [
    'FULL_NAME',
    'BIRTH_DATE',
    'CITY_NAME',
    'PROVINCE_NAME'
]

bnd = pd.read_csv(
    'Bank NIT DB.csv',
    encoding='utf-8-sig',
    usecols=columns_to_load
)

# نرمال‌سازی ستون تاریخ تولد
# حذف فاصله‌های اضافی
bnd['BIRTH_DATE'] = (
    bnd['BIRTH_DATE']
    .astype(str)
    .str.strip()
)

my_birthday = '1385-01-11'

# پیدا کردن تمامی افرادی که
# تاریخ تولدشان با کاربر یکسان است
twins = bnd[
    bnd['BIRTH_DATE'] == my_birthday
]

twins

# Question 11
Identifying customers whose father's name contains non-Arabic Persian characters such as گ، چ، پ or ژ.

In [ ]:
# =========================================
# Question 11
# =========================================

import pandas as pd

bnd = pd.read_csv(
    'Bank NIT DB.csv',
    encoding='utf-8-sig'
)

# پیدا کردن نام پدرهایی که شامل
# حروف گ، چ، پ یا ژ باشند
persian_mask = bnd['FATHER_NAME'].str.contains(
    r'[گچپژ]',
    na=False
)

# استخراج رکوردهای مورد نظر
persian_fathers = bnd[persian_mask]
persian_fathers

# Question 12
Calculating the total number of customers residing in Khalkhal city, Ardabil Province.

In [ ]:
# =========================================
# Question 12
# شمارش افراد ساکن شهر خلخال
# =========================================

import pandas as pd

bnd = pd.read_csv(
    'Bank NIT DB.csv',
    encoding='utf-8-sig'
)

# نرمال‌سازی نام استان و شهر
# برای جلوگیری از خطاهای یونیکدی
bnd['PROVINCE_NAME'] = (
    bnd['PROVINCE_NAME']
    .astype(str)
    .str.replace('ي', 'ی')
    .str.replace('ك', 'ک')
    .str.strip()
)

bnd['CITY_NAME'] = (
    bnd['CITY_NAME']
    .astype(str)
    .str.replace('ي', 'ی')
    .str.replace('ك', 'ک')
    .str.strip()
)

# ساخت فیلتر برای افراد ساکن خلخال
mask = (
    (bnd['PROVINCE_NAME'] == 'اردبیل') &
    (bnd['CITY_NAME'] == 'خلخال')
)

# استخراج تعداد افراد
khalkhal_count = len(bnd[mask])
khalkhal_count